# 03 · Diff & Passport：差异可见，护照可复现

**目标读者**：贡献者、审计者

**核心目的**：把「按主键对齐重建 diff 表格」与「护照字段契约」讲透 —— 删行后行号不能错位，6 种颜色各有确定语义，护照字段在 01 与 00 之间必须一字不差。

**验收标准**：

- 6 种 change_type 各有颜色与图例，颜色语义在 00 / 01 中一致；
- hover 显示 `rule_id + decision_id + reason`；
- 删行后 diff 表格与 clean 数据逐行对齐，不依赖原始行号；
- `passport.json` 含 `input_hash`、`output_file_sha256`、`human_decisions_count`、`unresolved_issues_count`、`auto_only`；
- 按护照 replay 可重建 clean 数据，且逐单元格一致。

## 大纲

1. 主键对齐：为什么不能用行号做定位
2. 6 种颜色语义与图例
3. HTML 生成与 hover 信息
4. 护照字段契约（与 01 保持一致）
5. replay 复现校验

In [1]:
# 1. 主键对齐：删行之后行号会错位，只有主键能定位
import json
import re
import sys
from pathlib import Path

import pandas as pd

import finpass as fp
from finpass import passport

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT / 'tests') not in sys.path:
    sys.path.insert(0, str(ROOT / 'tests'))
from _utils import normalize_frame  # 与测试共用同一份归一化，避免两套判据

OUT03 = ROOT / 'out' / '03'
OUT03.mkdir(parents=True, exist_ok=True)
DIRTY = ROOT / 'examples' / 'pingan_bank_daily_dirty.csv'
KEY = '日期'

DECISIONS = {
    'missing_token': ('answer', {}),
    'duplicate_key': ('answer', {}),
    'ohlc_contradiction': ('answer', {}),
    'unit_ambiguity': ('batch_apply', {'scale': 0.0001}),
    'date_format': ('answer', {}),
}

session = fp.open(DIRTY)
session.start()
while (question := session.next()) is not None:
    action, params = DECISIONS[question.rule_id]
    session.answer(question.question_id, action, params, '03 演示')
written = session.export(OUT03)
stem = DIRTY.stem

changes = passport.read_changes(written['changes'])
clean = pd.read_csv(written['clean'])
dropped = [c for c in changes if c['change_type'] == 'dropped_row']

print('原始行数', len(session.original), '→ clean 行数', len(clean), '| dropped_row', len(dropped))
assert len(clean) == len(session.original) - len(dropped)

# 位置对齐会错：删掉的是重复主键的第二次出现，此后的行整体前移一位
original_keys = session.original[KEY].astype(str).str.replace('/', '-', regex=False).tolist()
kept = [key for key, dup in zip(original_keys, session.original[KEY].duplicated(keep='first')) if not dup]
assert clean[KEY].astype(str).tolist() == kept
print('2024-03-19 在原始 CSV 是第 8 行，在 clean 里是第', kept.index('2024-03-19') + 1, '行')

原始行数 8 → clean 行数 7 | dropped_row 1
2024-03-19 在原始 CSV 是第 8 行，在 clean 里是第 7 行


In [2]:
# 2. 6 种颜色：图例常驻，颜色语义固定
from finpass.diff import COLOR_MAP

html = (OUT03 / f'{stem}.diff.html').read_text(encoding='utf-8')
for change_type, color in COLOR_MAP.items():
    assert color in html, (change_type, color)  # 图例把 6 种颜色都渲染出来

present = sorted({c['change_type'] for c in changes})
print('本次触发', present)
assert set(present) <= set(COLOR_MAP)
for change_type in present:
    if change_type == 'flagged_kept':
        continue  # 打标用橙色描边而非底色，避免与「已修改」的黄底混淆
    assert f'background:{COLOR_MAP[change_type]}' in html, change_type

print('本次未触发（由图例与单测覆盖）', sorted(set(COLOR_MAP) - set(present)))

本次触发 ['changed_value', 'dropped_row', 'filled_nan', 'normalized']
本次未触发（由图例与单测覆盖） ['dropped_col', 'flagged_kept']


In [3]:
# 3. hover：rule_id + decision_id + reason 三件套
events = [
    json.loads(line)
    for line in (OUT03 / f'{stem}.events.jsonl').read_text(encoding='utf-8').splitlines()
    if line.strip()
]
event_ids = {event['event_id'] for event in events}
rule_ids = {event['rule_id'] for event in events}
reasons = {event['event_id']: event['reason'] for event in events}

tags = re.findall(
    r'data-rule-id="([^"]+)"[^>]*data-decision-id="([0-9a-f]{32})"[^>]*'
    r'data-change-type="([^"]+)"[^>]*data-reason="([^"]*)"',
    html,
)
print('带 hover 的单元格数', len(tags))
assert tags
assert {rule for rule, _, _, _ in tags} <= rule_ids
assert {decision for _, decision, _, _ in tags} <= event_ids
assert all(reason == reasons[decision] for _, decision, _, reason in tags)

# 每条变更都要在页面上留锚点：锚点对不上，diff 就只是张图
anchored = {(rule, decision) for rule, decision, _, _ in tags}
expected = {(c['rule_id'], c['decision_id']) for c in changes if c['change_type'] != 'dropped_row'}
assert expected <= anchored, expected - anchored

带 hover 的单元格数 7


In [4]:
# 4. 护照字段契约：03 起多一个 output_file_sha256
payload = json.loads((OUT03 / f'{stem}.passport.json').read_text(encoding='utf-8'))
required = [
    'input_hash', 'output_file_sha256', 'input_path', 'keys',
    'profile_name', 'profile_version', 'human_decisions_count',
    'unresolved_issues_count', 'auto_only', 'cli_replay_cmd', 'reproduce',
]
missing = [field for field in required if field not in payload]
assert not missing, missing

# 护照里只有文件字节哈希：内存指纹已移除，避免两套判据
assert payload['output_file_sha256'] == passport.sha256_file(written['clean'])
assert payload['human_decisions_count'] == len(events)
assert payload['unresolved_issues_count'] == sum(1 for c in changes if c['change_type'] == 'flagged_kept')
assert payload['auto_only'] is False
print('护照字段齐全，计数与产物对得上')
print('reproduce =', payload['reproduce'])

护照字段齐全，计数与产物对得上
reproduce = finpass verify pingan_bank_daily_dirty.passport.json


In [5]:
# 5. replay：由原始数据 + changes 重建，与 clean.csv 逐单元格一致
after = json.loads((OUT03 / f'{stem}.report.after.json').read_text(encoding='utf-8'))
rebuilt = passport.replay_changes(session.original, changes, key=payload['keys'], dtypes=after['dtypes'])

# CSV 往返不保留 dtype：数值列先归一化，再逐单元格比
pd.testing.assert_frame_equal(normalize_frame(rebuilt), normalize_frame(clean))
print('replay 行数', len(rebuilt), '==', len(clean))

ok, messages = passport.verify_passport(written['passport'])
print(messages)
assert ok, messages
assert {'hash OK', 'replay OK', 'events OK'} <= set(messages)

replay 行数 7 == 7
['hash OK', 'replay OK', 'events OK']
